# Proverb Sentiment Pricer

Predicting a "wisdom score" (1–100) for Bible proverbs.

In [25]:
import random
import time
from pathlib import Path

import gradio as gr
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

load_dotenv(override=True)
client = OpenAI()

random.seed(42)
np.random.seed(42)

JSONL_DIR = Path("jsonl")
JSONL_DIR.mkdir(exist_ok=True)

## Load Pre-computed Data

In [26]:
from huggingface_hub import hf_hub_download
import json
import random

hf_dataset_id = "toluwalemi/scored_proverbs"
hf_dataset_filename = "scored_proverbs.json"

print(f"Downloading {hf_dataset_filename} from Hugging Face ({hf_dataset_id})...")
try:
    hf_file = hf_hub_download(
        repo_id=hf_dataset_id,
        filename=hf_dataset_filename,
        repo_type="dataset"
    )

    with open(hf_file, "r") as f:
        scored_proverbs = json.load(f)
    print(f"Successfully loaded {len(scored_proverbs)} scored proverbs from Hugging Face.")
except Exception as e:
    print(f"Error downloading from Hugging Face: {e}")
    scored_proverbs = []

# Shuffle and split
random.shuffle(scored_proverbs)
n = len(scored_proverbs)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_data = scored_proverbs[:train_end]
val_data = scored_proverbs[train_end:val_end]
test_data = scored_proverbs[val_end:]

print(f"Train: {len(train_data)}, Validation: {len(val_data)}, Test: {len(test_data)}")

Successfully loaded 881 scored proverbs from Hugging Face.
Train: 616, Validation: 132, Test: 133


## Evaluation Framework & Baselines


In [27]:
def evaluate(predictor_fn, data, label="Model"):
    actuals = []
    predictions = []
    for p in data:
        actual = p["score"]
        try:
            predicted = float(predictor_fn(p))
            predicted = max(1, min(100, predicted))
        except:
            predicted = 50.0
        actuals.append(actual)
        predictions.append(predicted)

    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    r2 = r2_score(actuals, predictions)

    print(f"--- {label} ---")
    print(f"  MAE:  {mae:.2f} points | RMSE: {rmse:.2f} points | R²: {r2:.4f}\n")
    return predicted

In [28]:
# Baseline 1: Random
def random_scorer(proverb):
    return random.randint(1, 100)
evaluate(random_scorer, test_data, label="Random Baseline")

# Baseline 2: Average Score
training_avg = np.mean([p["score"] for p in train_data])
def constant_scorer(proverb):
    return training_avg
evaluate(constant_scorer, test_data, label="Constant (Average) Baseline")

# Baseline 3: BoW + Linear Regression
train_texts = [p["text"] for p in train_data]
vectorizer = CountVectorizer(max_features=500, stop_words="english")
X_train_bow = vectorizer.fit_transform(train_texts)
train_scores = np.array([p["score"] for p in train_data])

bow_model = LinearRegression()
bow_model.fit(X_train_bow, train_scores)

def bow_lr_scorer(proverb):
    x = vectorizer.transform([proverb["text"]])
    return bow_model.predict(x)[0]
evaluate(bow_lr_scorer, test_data, label="BoW + Linear Regression")

--- Random Baseline ---
  MAE:  35.05 points | RMSE: 43.20 points | R²: -5.5480

--- Constant (Average) Baseline ---
  MAE:  12.12 points | RMSE: 16.88 points | R²: -0.0001

--- BoW + Linear Regression ---
  MAE:  18.58 points | RMSE: 24.62 points | R²: -1.1273



100

## Fine-Tuned Model
Checks OpenAI for a completed 'proverb' model.
If not found, upload the data, launch the job, and wait for the results.

In [29]:
fine_tuned_model_name = None
fine_tuned_scorer = None

def ft_messages_for(proverb):
    user_msg = (
        f"Rate the wisdom value of this proverb on a scale of 1 to 100. "
        f"Respond with ONLY the number.\n\n{proverb['text']}"
    )
    return [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": str(proverb["score"])}
    ]

def make_jsonl(items):
    return "\n".join([json.dumps({"messages": ft_messages_for(item)}) for item in items])

# Check if there is a successful proverb fine-tuning job
try:
    all_jobs = client.fine_tuning.jobs.list(limit=10).data
    for j in all_jobs:
        if j.status == "succeeded" and j.fine_tuned_model and "proverb" in j.fine_tuned_model:
            fine_tuned_model_name = j.fine_tuned_model
            break
except Exception as e:
    print(f"Could not query fine-tuning jobs: {e}")

if fine_tuned_model_name:
    print(f"Found existing pre-trained model on your account: {fine_tuned_model_name}")
    print("Skipping training data upload and fine-tuning job creation.")
else:
    print("No completed fine-tuning job found. Initiating upload and training sequence...")
    
    # Download fine-tuning data from Hugging Face
    print("Downloading ft_proverb_train.jsonl and ft_proverb_val.jsonl from Hugging Face...")
    from huggingface_hub import hf_hub_download
    train_path = hf_hub_download(repo_id="toluwalemi/scored_proverbs", filename="ft_proverb_train.jsonl", repo_type="dataset")
    val_path = hf_hub_download(repo_id="toluwalemi/scored_proverbs", filename="ft_proverb_val.jsonl", repo_type="dataset")
    
    print(f"Uploading training and validation examples to OpenAI for fine-tuning...")
    
    with open(train_path, "rb") as f: train_file = client.files.create(file=f, purpose="fine-tune")
    with open(val_path, "rb") as f: val_file = client.files.create(file=f, purpose="fine-tune")
    
    print(f"Creating fine-tuning job...")
    ft_job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model="gpt-4.1-nano-2025-04-14",
        seed=42,
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        suffix="proverb"
    )
    
    print(f"Job created: {ft_job.id}. Waiting for completion... (This takes 2-5 minutes)")
    
    while True:
        job = client.fine_tuning.jobs.retrieve(ft_job.id)
        if job.status == "succeeded":
            fine_tuned_model_name = job.fine_tuned_model
            print(f"\nFine-tuning complete! Model: {fine_tuned_model_name}")
            break
        elif job.status in ["failed", "cancelled"]:
            print(f"\nJob {job.status}.")
            break
        else:
            print(".", end="", flush=True)
            time.sleep(10)

if fine_tuned_model_name:
    def ft_scorer(proverb):
        user_msg = (
            f"Rate the wisdom value of this proverb on a scale of 1 to 100. "
            f"Respond with ONLY the number.\n\n{proverb['text']}"
        )
        response = client.chat.completions.create(
            model=fine_tuned_model_name,
            messages=[{"role": "user", "content": user_msg}],
            max_tokens=5,
            temperature=0.1
        )
        import re
        return float(re.sub(r"[^0-9.]", "", response.choices[0].message.content))
        
    fine_tuned_scorer = ft_scorer


Found existing pre-trained model on your account: ft:gpt-4.1-nano-2025-04-14:personal:proverb-scorer:DGPTPovO
Skipping training data upload and fine-tuning job creation.


## Gradio


In [30]:
def score_proverb(proverb_text, model_choice):
    proverb = {"text": proverb_text, "chapter": 0, "reference": "Custom"}

    if model_choice == "Random Baseline":
        score = random.randint(1, 100)
        method = "Random number between 1-100"
    elif model_choice == "Average Baseline":
        score = training_avg
        method = f"Always predicts the training average: {training_avg:.1f}"
    elif model_choice == "BoW + Linear Regression":
        x = vectorizer.transform([proverb_text])
        score = max(1, min(100, bow_model.predict(x)[0]))
        method = "Bag-of-words features + linear regression"
    elif model_choice == "GPT-4.1-nano (Fine-Tuned)" and fine_tuned_scorer:
        try:
            score = fine_tuned_scorer(proverb)
            method = "Fine-tuned on our curated proverb scoring dataset"
        except Exception as e:
            return f"Model Error: {str(e)}"
    else:
        return "Model not available. Select another model."

    score = float(score)
    tier = "Legendary" if score >= 90 else "Premium" if score >= 70 else "Solid" if score >= 50 else "Niche" if score >= 30 else "Fragment"

    return f"**Wisdom Score: {score:.1f}/100**\n\n**Tier:** {tier}\n\n**Method:** {method}"

model_choices = ["Random Baseline", "Average Baseline", "BoW + Linear Regression"]
default_model = "BoW + Linear Regression"
if fine_tuned_scorer:
    model_choices.append("GPT-4.1-nano (Fine-Tuned)")
    default_model = "GPT-4.1-nano (Fine-Tuned)"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Proverb Wisdom Scorer")
    with gr.Row():
        with gr.Column(scale=2):
            proverb_input = gr.Textbox(label="Enter a proverb", lines=3)
            score_btn = gr.Button("Score This Proverb", variant="primary")
        with gr.Column(scale=1):
            model_dropdown = gr.Dropdown(choices=model_choices, value=default_model, label="Model")
            single_output = gr.Markdown(label="Result")

    score_btn.click(fn=score_proverb, inputs=[proverb_input, model_dropdown], outputs=single_output)

demo.launch(inline=False, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
